In [1]:
pip install "numpy<2" --force-reinstall

  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepnote-toolkit 1.1.2 requires pandas<2.2,>=1.2.5; python_version < "3.12", but you have pandas 2.3.3 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
#!/usr/bin/env python3
"""
3_ANALYSIS_LLM_PREDICTIONS.py

Comprehensive analysis of LLM predictions of pluralistic ignorance across 8 stages and 125 countries.

Analyses:
1. Descriptive statistics by stage and model
2. Correlation with ground truth (if available)
3. Stage effects - does more information improve accuracy?
4. Model comparisons - which models perform best?
5. Cross-country patterns - which countries are easiest/hardest to predict?
6. Information type analysis - which info matters most?
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend for matplotlib 3.9+
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Set visualization style - FIXED for matplotlib 3.9+
sns.set_style("whitegrid")
plt.rcParams.update({
    'font.size': 10,
    'axes.grid': True
})

print("="*80)
print("LLM PLURALISTIC IGNORANCE PREDICTION ANALYSIS")
print("="*80)

# ================================================================
# Load Data
# ================================================================

print("\n1. Loading data...")
df = pd.read_csv("predictions_all_stages_long.csv")
print(f"   ✓ Loaded {len(df)} rows")
print(f"   ✓ Countries: {df['countrynew'].nunique()}")
print(f"   ✓ Stages: {sorted(df['stage'].unique())}")
print(f"   ✓ Models: {[col.replace('pred_', '') for col in df.columns if col.startswith('pred_')]}")

# Create ensemble prediction (average of all models)
model_cols = [col for col in df.columns if col.startswith('pred_')]
df['pred_ensemble'] = df[model_cols].mean(axis=1)
print(f"   ✓ Created ensemble prediction (mean of {len(model_cols)} models)")

# ================================================================
# Add Continent Information
# ================================================================

# Country to continent mapping
continent_mapping = {
    'Afghanistan': 'Asia', 'Albania': 'Europe', 'Algeria': 'Africa', 'Argentina': 'South America',
    'Armenia': 'Asia', 'Australia': 'Oceania', 'Austria': 'Europe', 'Bangladesh': 'Asia',
    'Belgium': 'Europe', 'Benin': 'Africa', 'Bolivia': 'South America', 'Bosnia Herzegovina': 'Europe',
    'Botswana': 'Africa', 'Brazil': 'South America', 'Bulgaria': 'Europe', 'Burkina Faso': 'Africa',
    'Cambodia': 'Asia', 'Cameroon': 'Africa', 'Canada': 'North America', 'Chad': 'Africa',
    'Chile': 'South America', 'China': 'Asia', 'Colombia': 'South America', 'Congo Brazzaville': 'Africa',
    'Costa Rica': 'North America', 'Croatia': 'Europe', 'Cyprus': 'Europe', 'Czech Republic': 'Europe',
    'Denmark': 'Europe', 'Dominican Republic': 'North America', 'Ecuador': 'South America', 'Egypt': 'Africa',
    'El Salvador': 'North America', 'Estonia': 'Europe', 'Ethiopia': 'Africa', 'Finland': 'Europe',
    'France': 'Europe', 'Gabon': 'Africa', 'Georgia': 'Asia', 'Germany': 'Europe',
    'Ghana': 'Africa', 'Greece': 'Europe', 'Guatemala': 'North America', 'Guinea': 'Africa',
    'Haiti': 'North America', 'Honduras': 'North America', 'Hong Kong': 'Asia', 'Hungary': 'Europe',
    'Iceland': 'Europe', 'India': 'Asia', 'Indonesia': 'Asia', 'Iran': 'Asia',
    'Iraq': 'Asia', 'Ireland': 'Europe', 'Israel': 'Asia', 'Italy': 'Europe',
    'Ivory Coast': 'Africa', 'Jamaica': 'North America', 'Japan': 'Asia', 'Jordan': 'Asia',
    'Kazakhstan': 'Asia', 'Kenya': 'Africa', 'Kosovo': 'Europe', 'Kyrgyzstan': 'Asia',
    'Laos': 'Asia', 'Latvia': 'Europe', 'Lebanon': 'Asia', 'Liberia': 'Africa', 'Libya': 'Africa',
    'Lithuania': 'Europe', 'Luxembourg': 'Europe', 'Macedonia': 'Europe', 'Madagascar': 'Africa',
    'Malawi': 'Africa', 'Malaysia': 'Asia', 'Mali': 'Africa', 'Malta': 'Europe',
    'Mauritania': 'Africa', 'Mauritius': 'Africa', 'Mexico': 'North America', 'Moldova': 'Europe',
    'Mongolia': 'Asia', 'Montenegro': 'Europe', 'Morocco': 'Africa', 'Mozambique': 'Africa',
    'Myanmar': 'Asia', 'Namibia': 'Africa', 'Nepal': 'Asia', 'Netherlands': 'Europe', 
    'New Zealand': 'Oceania', 'Nicaragua': 'North America', 'Niger': 'Africa', 'Nigeria': 'Africa',
    'North Macedonia': 'Europe', 'Norway': 'Europe', 'Pakistan': 'Asia', 'Palestinian Territories': 'Asia',
    'Panama': 'North America', 'Paraguay': 'South America', 'Peru': 'South America', 'Philippines': 'Asia',
    'Poland': 'Europe', 'Portugal': 'Europe', 'Romania': 'Europe', 'Russia': 'Europe', 'Rwanda': 'Africa',
    'Saudi Arabia': 'Asia', 'Senegal': 'Africa', 'Serbia': 'Europe', 'Sierra Leone': 'Africa',
    'Singapore': 'Asia', 'Slovakia': 'Europe', 'Slovenia': 'Europe', 'South Africa': 'Africa',
    'South Korea': 'Asia', 'Spain': 'Europe', 'Sri Lanka': 'Asia', 'Sweden': 'Europe',
    'Switzerland': 'Europe', 'Taiwan': 'Asia', 'Tajikistan': 'Asia', 'Tanzania': 'Africa',
    'Thailand': 'Asia', 'Togo': 'Africa', 'Tunisia': 'Africa', 'Turkey': 'Asia',
    'Turkmenistan': 'Asia', 'Uganda': 'Africa', 'Ukraine': 'Europe', 'United Arab Emirates': 'Asia',
    'United Kingdom': 'Europe', 'United States': 'North America', 'Uruguay': 'South America',
    'Uzbekistan': 'Asia', 'Venezuela': 'South America', 'Vietnam': 'Asia', 'Yemen': 'Asia',
    'Zambia': 'Africa', 'Zimbabwe': 'Africa'
}

df['continent'] = df['countrynew'].map(continent_mapping)

# Check for unmapped countries
unmapped = df[df['continent'].isna()]['countrynew'].unique()
if len(unmapped) > 0:
    print(f"   ⚠️  Warning: {len(unmapped)} countries not mapped to continents: {list(unmapped)}")
else:
    print(f"   ✓ All countries mapped to continents")

print(f"   ✓ Continents: {sorted(df['continent'].dropna().unique())}")
print(f"   ✓ Countries by continent:")
for continent in sorted(df['continent'].dropna().unique()):
    n_countries = df[df['continent'] == continent]['countrynew'].nunique()
    print(f"      {continent}: {n_countries} countries")

# ================================================================
# 2. Descriptive Statistics
# ================================================================

print("\n" + "="*80)
print("2. DESCRIPTIVE STATISTICS")
print("="*80)

print("\n2.1 Predictions by Stage:")
stage_stats = df.groupby('stage')[model_cols + ['pred_ensemble']].agg(['mean', 'std', 'min', 'max'])
print(stage_stats.round(2))

print("\n2.2 Predictions by Model (across all stages):")
model_stats = df[model_cols + ['pred_ensemble']].agg(['mean', 'std', 'min', 'max'])
print(model_stats.round(2))

print("\n2.3 Missing Data Check:")
for col in model_cols:
    missing = df[col].isna().sum()
    pct = (missing / len(df)) * 100
    print(f"   {col}: {missing} missing ({pct:.1f}%)")

# ================================================================
# 3. Load Ground Truth (if available)
# ================================================================

print("\n" + "="*80)
print("3. GROUND TRUTH COMPARISON")
print("="*80)

try:
    # Try to load the original data with ground truth
    gt_df = pd.read_csv("data_final.csv")
    
    # Merge with predictions
    if 'mean_other_willingness' in gt_df.columns:
        gt_df['ground_truth_pi'] = gt_df['mean_other_willingness'] * 100  # Convert to percentage
        
        # Merge ground truth into predictions
        df = df.merge(
            gt_df[['countrynew', 'ground_truth_pi', 'mean_own_willingness']], 
            on='countrynew', 
            how='left'
        )
        
        print(f"   ✓ Ground truth loaded for {df['ground_truth_pi'].notna().sum()} country-stage combinations")
        print(f"   ✓ Ground truth mean: {df['ground_truth_pi'].mean():.2f}%")
        print(f"   ✓ Ground truth std: {df['ground_truth_pi'].std():.2f}%")
        
        has_ground_truth = True
    else:
        print("   ⚠️  'mean_other_willingness' not found in data_final.csv")
        has_ground_truth = False
        
except FileNotFoundError:
    print("   ⚠️  data_final.csv not found - skipping ground truth analysis")
    has_ground_truth = False

# ================================================================
# 4. Accuracy Analysis (if ground truth available)
# ================================================================

if has_ground_truth:
    print("\n" + "="*80)
    print("4. ACCURACY ANALYSIS")
    print("="*80)
    
    # Calculate errors for each model
    for col in model_cols + ['pred_ensemble']:
        model_name = col.replace('pred_', '')
        
        # Mean Absolute Error
        df[f'mae_{model_name}'] = abs(df[col] - df['ground_truth_pi'])
        
        # Signed Error (bias)
        df[f'error_{model_name}'] = df[col] - df['ground_truth_pi']
    
    print("\n4.1 Mean Absolute Error (MAE) by Model:")
    mae_cols = [col for col in df.columns if col.startswith('mae_')]
    mae_stats = df[mae_cols].agg(['mean', 'std']).T
    mae_stats.columns = ['MAE', 'SD']
    mae_stats.index = [col.replace('mae_', '') for col in mae_stats.index]
    mae_stats = mae_stats.sort_values('MAE')
    print(mae_stats.round(2))
    
    print("\n4.2 Bias (Mean Signed Error) by Model:")
    error_cols = [col for col in df.columns if col.startswith('error_') and not col.startswith('error_mae')]
    bias_stats = df[error_cols].mean().to_frame('Bias')
    bias_stats.index = [col.replace('error_', '') for col in bias_stats.index]
    bias_stats = bias_stats.sort_values('Bias')
    print(bias_stats.round(2))
    print("\nNote: Positive bias = overestimation, Negative bias = underestimation")
    
    print("\n4.3 Correlations with Ground Truth:")
    corr_results = []
    for col in model_cols + ['pred_ensemble']:
        model_name = col.replace('pred_', '')
        valid_data = df[[col, 'ground_truth_pi']].dropna()
        
        if len(valid_data) > 0:
            pearson_r, pearson_p = pearsonr(valid_data[col], valid_data['ground_truth_pi'])
            spearman_r, spearman_p = spearmanr(valid_data[col], valid_data['ground_truth_pi'])
            
            corr_results.append({
                'Model': model_name,
                'Pearson r': pearson_r,
                'Pearson p': pearson_p,
                'Spearman ρ': spearman_r,
                'Spearman p': spearman_p,
                'N': len(valid_data)
            })
    
    corr_df = pd.DataFrame(corr_results)
    corr_df = corr_df.sort_values('Pearson r', ascending=False)
    print(corr_df.round(4))
    
    print("\n4.4 MAE by Stage (shows if more info improves accuracy):")
    stage_mae = df.groupby('stage')[[f'mae_{m}' for m in ['gpt', 'claude', 'gemini', 'llama', 'ensemble']]].mean()
    stage_mae.columns = [col.replace('mae_', '') for col in stage_mae.columns]
    print(stage_mae.round(2))

# ================================================================
# 5. Stage Effects Analysis
# ================================================================

print("\n" + "="*80)
print("5. STAGE EFFECTS ANALYSIS")
print("="*80)

print("\n5.1 Prediction Changes Across Stages:")
stage_changes = df.groupby('stage')['pred_ensemble'].agg(['mean', 'std', 'min', 'max'])
print(stage_changes.round(2))

print("\n5.2 Stage-to-Stage Changes (mean absolute change):")
for stage in range(2, 9):
    prev_stage = df[df['stage'] == stage - 1].set_index('countrynew')['pred_ensemble']
    curr_stage = df[df['stage'] == stage].set_index('countrynew')['pred_ensemble']
    
    # Align by country
    common_countries = prev_stage.index.intersection(curr_stage.index)
    change = abs(curr_stage.loc[common_countries] - prev_stage.loc[common_countries]).mean()
    
    print(f"   Stage {stage-1} → {stage}: {change:.2f} percentage points")

print("\n5.3 Information Type Labels:")
stage_info = {
    1: "Country only",
    2: "+ Socio-demographics",
    3: "+ Macro-economics",
    4: "+ Temperature",
    5: "+ Own willingness",
    6: "+ Socio + Macro",
    7: "+ Socio + Macro + Temp",
    8: "+ All info"
}
for stage, info in stage_info.items():
    mean_pred = df[df['stage'] == stage]['pred_ensemble'].mean()
    print(f"   Stage {stage} ({info}): {mean_pred:.2f}%")

# ================================================================
# 6. Model Comparison
# ================================================================

print("\n" + "="*80)
print("6. MODEL COMPARISON")
print("="*80)

print("\n6.1 Pairwise Correlations Between Models:")
model_data = df[model_cols].dropna()
corr_matrix = model_data.corr()
print(corr_matrix.round(3))

print("\n6.2 Model Agreement (mean absolute difference between models):")
agreement_results = []
models = ['gpt', 'claude', 'gemini', 'llama']
for i, m1 in enumerate(models):
    for m2 in models[i+1:]:
        diff = abs(df[f'pred_{m1}'] - df[f'pred_{m2}']).mean()
        agreement_results.append({
            'Model Pair': f"{m1} vs {m2}",
            'Mean Abs Diff': diff
        })

agreement_df = pd.DataFrame(agreement_results)
agreement_df = agreement_df.sort_values('Mean Abs Diff')
print(agreement_df.round(2))

print("\n6.3 Model Variance (how much does each model vary?):")
for col in model_cols:
    model_name = col.replace('pred_', '')
    variance = df[col].std()
    print(f"   {model_name}: SD = {variance:.2f}")

# ================================================================
# 7. Country-Level Analysis
# ================================================================

print("\n" + "="*80)
print("7. COUNTRY-LEVEL ANALYSIS")
print("="*80)

print("\n7.1 Easiest vs Hardest Countries to Predict (by model agreement):")
country_agreement = df.groupby('countrynew')[model_cols].std().mean(axis=1).sort_values()
print("\nMost Agreement (easiest):")
print(country_agreement.head(10).round(2))
print("\nLeast Agreement (hardest):")
print(country_agreement.tail(10).round(2))

if has_ground_truth:
    print("\n7.2 Most vs Least Accurate Predictions:")
    country_mae = df.groupby('countrynew')['mae_ensemble'].mean().sort_values()
    print("\nMost Accurate:")
    print(country_mae.head(10).round(2))
    print("\nLeast Accurate:")
    print(country_mae.tail(10).round(2))

print("\n7.3 Countries with Largest Stage Effects:")
country_stage_variance = df.groupby('countrynew')['pred_ensemble'].std().sort_values(ascending=False)
print("\nMost Sensitive to Information:")
print(country_stage_variance.head(10).round(2))
print("\nLeast Sensitive to Information:")
print(country_stage_variance.tail(10).round(2))

# ================================================================
# 7B. Continent-Level Analysis
# ================================================================

print("\n" + "="*80)
print("7B. CONTINENT-LEVEL ANALYSIS")
print("="*80)

if 'continent' in df.columns and df['continent'].notna().any():
    print("\n7B.1 Mean Predictions by Continent:")
    continent_pred = df.groupby('continent')['pred_ensemble'].agg(['mean', 'std', 'count'])
    continent_pred.columns = ['Mean Prediction', 'SD', 'N (country-stages)']
    continent_pred = continent_pred.sort_values('Mean Prediction', ascending=False)
    print(continent_pred.round(2))
    
    if has_ground_truth:
        print("\n7B.2 Mean Absolute Error (MAE) by Continent:")
        continent_mae = df.groupby('continent').agg({
            'mae_gpt': 'mean',
            'mae_claude': 'mean',
            'mae_gemini': 'mean',
            'mae_llama': 'mean',
            'mae_ensemble': 'mean'
        })
        continent_mae.columns = ['MAE_GPT', 'MAE_Claude', 'MAE_Gemini', 'MAE_Llama', 'MAE_Ensemble']
        continent_mae = continent_mae.sort_values('MAE_Ensemble')
        print(continent_mae.round(2))
        
        print("\n7B.3 Accuracy Ranking by Continent (Ensemble):")
        print("(Lower MAE = Better)")
        continent_ranking = continent_mae[['MAE_Ensemble']].sort_values('MAE_Ensemble')
        for rank, (continent, row) in enumerate(continent_ranking.iterrows(), 1):
            print(f"   {rank}. {continent}: MAE = {row['MAE_Ensemble']:.2f}pp")
        
        print("\n7B.4 Ground Truth Statistics by Continent:")
        continent_gt = df.groupby('continent')['ground_truth_pi'].agg(['mean', 'std', 'min', 'max'])
        continent_gt.columns = ['Mean GT', 'SD GT', 'Min GT', 'Max GT']
        print(continent_gt.round(2))
        
        print("\n7B.5 Correlation with Ground Truth by Continent:")
        continent_corr = []
        for continent in df['continent'].dropna().unique():
            continent_data = df[df['continent'] == continent][['pred_ensemble', 'ground_truth_pi']].dropna()
            if len(continent_data) > 10:  # Only if enough data
                r, p = pearsonr(continent_data['pred_ensemble'], continent_data['ground_truth_pi'])
                continent_corr.append({
                    'Continent': continent,
                    'Pearson r': r,
                    'p-value': p,
                    'N': len(continent_data)
                })
        
        if continent_corr:
            continent_corr_df = pd.DataFrame(continent_corr)
            continent_corr_df = continent_corr_df.sort_values('Pearson r', ascending=False)
            print(continent_corr_df.round(4))
        
        print("\n7B.6 Bias (Over/Under-estimation) by Continent:")
        continent_bias = df.groupby('continent').agg({
            'error_gpt': 'mean',
            'error_claude': 'mean',
            'error_gemini': 'mean',
            'error_llama': 'mean',
            'error_ensemble': 'mean'
        })
        continent_bias.columns = ['Bias_GPT', 'Bias_Claude', 'Bias_Gemini', 'Bias_Llama', 'Bias_Ensemble']
        print(continent_bias.round(2))
        print("\nNote: Positive = overestimation, Negative = underestimation")
    
    print("\n7B.7 Model Agreement by Continent:")
    continent_agreement = []
    for continent in df['continent'].dropna().unique():
        continent_data = df[df['continent'] == continent][model_cols]
        # Calculate standard deviation across models for each prediction
        agreement = continent_data.std(axis=1).mean()
        continent_agreement.append({
            'Continent': continent,
            'Mean Model Disagreement': agreement,
            'N': len(continent_data)
        })
    
    continent_agreement_df = pd.DataFrame(continent_agreement)
    continent_agreement_df = continent_agreement_df.sort_values('Mean Model Disagreement')
    print(continent_agreement_df.round(2))
    print("\nNote: Lower disagreement = models more consistent")
    
    print("\n7B.8 Stage Effects by Continent:")
    print("(How much predictions change across stages)")
    continent_stage_effects = df.groupby('continent')['pred_ensemble'].std()
    continent_stage_effects = continent_stage_effects.sort_values(ascending=False)
    print(continent_stage_effects.round(2))
    
    # ================================================================
    # 7B.9 VISUALIZATION: MAE by Continent with 95% CI
    # ================================================================
    
    if has_ground_truth:
        print("\n7B.9 Creating MAE by Continent Visualization...")
        
        from scipy import stats as scipy_stats
        
        fig, axes = plt.subplots(2, 2)
        fig.set_size_inches(16, 10)
        fig.set_dpi(100)
        fig.suptitle('Mean Absolute Error by Continent (with 95% CI)', fontsize=16, fontweight='bold')
        
        models = ['gpt', 'claude', 'gemini', 'llama']
        
        for idx, model in enumerate(models):
            row = idx // 2
            col = idx % 2
            ax = axes[row, col]
            
            mae_col = f'mae_{model}'
            
            # Get unique continents
            continents = sorted(df['continent'].dropna().unique())
            
            means = []
            cis_lower = []
            cis_upper = []
            continent_labels = []
            
            for continent in continents:
                continent_data = df[df['continent'] == continent][mae_col].dropna()
                
                if len(continent_data) > 0:
                    mean_error = continent_data.mean()
                    
                    # Calculate 95% confidence interval
                    sem = scipy_stats.sem(continent_data)
                    ci = sem * scipy_stats.t.ppf((1 + 0.95) / 2, len(continent_data) - 1)
                    
                    means.append(mean_error)
                    cis_lower.append(ci)
                    cis_upper.append(ci)
                    continent_labels.append(continent)
            
            # Create bar plot
            x_pos = np.arange(len(continent_labels))
            bars = ax.bar(x_pos, means, alpha=0.7, color='steelblue', edgecolor='black', linewidth=1)
            
            # Add error bars
            ax.errorbar(x_pos, means, yerr=[cis_lower, cis_upper], 
                       fmt='none', ecolor='black', capsize=5, capthick=2, linewidth=2)
            
            # Add value labels on bars
            for i, (bar, mean) in enumerate(zip(bars, means)):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + cis_upper[i] + 0.5,
                       f'{mean:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
            
            ax.set_ylabel('Mean Absolute Error (pp)', fontweight='bold', fontsize=11)
            ax.set_xlabel('Continent', fontweight='bold', fontsize=11)
            ax.set_title(f'{model.upper()}', fontweight='bold', fontsize=12)
            ax.set_xticks(x_pos)
            ax.set_xticklabels(continent_labels, rotation=45, ha='right')
            ax.grid(True, alpha=0.3, axis='y')
            ax.set_ylim(bottom=0)
        
        plt.tight_layout()
        plt.savefig('mae_by_continent_with_ci.png', dpi=300, bbox_inches='tight')
        plt.savefig('mae_by_continent_with_ci.pdf', dpi=300, bbox_inches='tight')
        print("   ✓ Saved mae_by_continent_with_ci.png")
        print("   ✓ Saved mae_by_continent_with_ci.pdf")
        plt.close()
        
        # Save continent statistics with CI
        continent_ci_stats = []
        for model in models:
            mae_col = f'mae_{model}'
            for continent in continents:
                continent_data = df[df['continent'] == continent][mae_col].dropna()
                if len(continent_data) > 0:
                    mean_error = continent_data.mean()
                    sem = scipy_stats.sem(continent_data)
                    ci = sem * scipy_stats.t.ppf((1 + 0.95) / 2, len(continent_data) - 1)
                    
                    continent_ci_stats.append({
                        'Model': model.upper(),
                        'Continent': continent,
                        'N': len(continent_data),
                        'Mean_MAE': mean_error,
                        'Std_Error': sem,
                        'CI_95': ci,
                        'CI_Lower': mean_error - ci,
                        'CI_Upper': mean_error + ci
                    })
        
        continent_ci_df = pd.DataFrame(continent_ci_stats)
        continent_ci_df.to_csv('mae_by_continent_with_ci_stats.csv', index=False)
        print("   ✓ Saved mae_by_continent_with_ci_stats.csv")
    
    # ================================================================
    # 7B.10 STATISTICAL CLUSTERING BY MAE
    # ================================================================
    
    if has_ground_truth:
        print("\n7B.10 Statistical Clustering by MAE...")
        
        from sklearn.cluster import KMeans
        from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
        from sklearn.preprocessing import StandardScaler
        
        # Get country-level average MAE across all stages
        country_mae_summary = df.groupby('countrynew').agg({
            'mae_ensemble': 'mean',
            'mae_gpt': 'mean',
            'mae_claude': 'mean',
            'mae_gemini': 'mean',
            'mae_llama': 'mean',
            'continent': 'first'
        }).reset_index()
        
        # Method 1: Jenks Natural Breaks (Optimal binning)
        print("\n   Method 1: Jenks Natural Breaks Optimization")
        try:
            from jenkspy import jenks_breaks
            
            mae_values = country_mae_summary['mae_ensemble'].values
            
            # Find natural breaks for 3 groups (Low, Medium, High)
            breaks = jenks_breaks(mae_values, n_classes=3)
            
            print(f"      Natural break thresholds: {[f'{b:.2f}' for b in breaks]}")
            
            # Assign clusters
            country_mae_summary['mae_cluster_jenks'] = pd.cut(
                country_mae_summary['mae_ensemble'],
                bins=breaks,
                labels=['Low MAE', 'Medium MAE', 'High MAE'],
                include_lowest=True
            )
            
            print(f"      Low MAE: < {breaks[1]:.2f}pp")
            print(f"      Medium MAE: {breaks[1]:.2f} - {breaks[2]:.2f}pp")
            print(f"      High MAE: > {breaks[2]:.2f}pp")
            
            # Show cluster sizes
            cluster_sizes = country_mae_summary['mae_cluster_jenks'].value_counts()
            print(f"\n      Cluster sizes:")
            for cluster, size in cluster_sizes.items():
                print(f"         {cluster}: {size} countries")
            
            has_jenks = True
            
        except ImportError:
            print("      ⚠️  jenkspy not available, skipping Jenks breaks")
            has_jenks = False
        
        # Method 2: K-Means Clustering
        print("\n   Method 2: K-Means Clustering (k=3)")
        
        # Use ensemble MAE for clustering
        X = country_mae_summary[['mae_ensemble']].values
        
        kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
        country_mae_summary['mae_cluster_kmeans'] = kmeans.fit_predict(X)
        
        # Get cluster centers and sort
        cluster_centers = kmeans.cluster_centers_.flatten()
        cluster_order = np.argsort(cluster_centers)
        
        # Relabel clusters as Low/Medium/High based on centers
        cluster_mapping = {cluster_order[0]: 'Low MAE', 
                          cluster_order[1]: 'Medium MAE', 
                          cluster_order[2]: 'High MAE'}
        country_mae_summary['mae_cluster_kmeans'] = country_mae_summary['mae_cluster_kmeans'].map(cluster_mapping)
        
        print(f"      Cluster centers: {sorted(cluster_centers)}")
        
        for cluster_name in ['Low MAE', 'Medium MAE', 'High MAE']:
            cluster_data = country_mae_summary[country_mae_summary['mae_cluster_kmeans'] == cluster_name]
            print(f"      {cluster_name}: {len(cluster_data)} countries, mean MAE = {cluster_data['mae_ensemble'].mean():.2f}pp")
        
        # Method 3: Hierarchical Clustering with Optimal Cut
        print("\n   Method 3: Hierarchical Clustering")
        
        # Perform hierarchical clustering
        Z = linkage(X, method='ward')
        
        # Cut tree to get 3 clusters
        hierarchical_labels = fcluster(Z, 3, criterion='maxclust')
        country_mae_summary['mae_cluster_hierarchical'] = hierarchical_labels
        
        # Get cluster means and sort
        cluster_means = country_mae_summary.groupby('mae_cluster_hierarchical')['mae_ensemble'].mean().sort_values()
        cluster_mapping_hier = {cluster_means.index[0]: 'Low MAE',
                               cluster_means.index[1]: 'Medium MAE',
                               cluster_means.index[2]: 'High MAE'}
        country_mae_summary['mae_cluster_hierarchical'] = country_mae_summary['mae_cluster_hierarchical'].map(cluster_mapping_hier)
        
        for cluster_name in ['Low MAE', 'Medium MAE', 'High MAE']:
            cluster_data = country_mae_summary[country_mae_summary['mae_cluster_hierarchical'] == cluster_name]
            print(f"      {cluster_name}: {len(cluster_data)} countries, mean MAE = {cluster_data['mae_ensemble'].mean():.2f}pp")
        
        # Method 4: Statistical Quartile/Percentile-based
        print("\n   Method 4: Percentile-based (Terciles)")
        
        country_mae_summary['mae_cluster_percentile'] = pd.qcut(
            country_mae_summary['mae_ensemble'],
            q=3,
            labels=['Low MAE', 'Medium MAE', 'High MAE']
        )
        
        # Get the percentile thresholds
        percentiles = country_mae_summary['mae_ensemble'].quantile([0, 1/3, 2/3, 1])
        print(f"      33rd percentile: {percentiles[1/3]:.2f}pp")
        print(f"      67th percentile: {percentiles[2/3]:.2f}pp")
        
        for cluster_name in ['Low MAE', 'Medium MAE', 'High MAE']:
            cluster_data = country_mae_summary[country_mae_summary['mae_cluster_percentile'] == cluster_name]
            print(f"      {cluster_name}: {len(cluster_data)} countries")
        
        # Method 5: Standard Deviation-based
        print("\n   Method 5: Standard Deviation-based")
        
        mae_mean = country_mae_summary['mae_ensemble'].mean()
        mae_std = country_mae_summary['mae_ensemble'].std()
        
        print(f"      Mean MAE: {mae_mean:.2f}pp")
        print(f"      Std Dev: {mae_std:.2f}pp")
        
        def classify_by_std(mae):
            if mae < mae_mean - 0.5 * mae_std:
                return 'Low MAE'
            elif mae > mae_mean + 0.5 * mae_std:
                return 'High MAE'
            else:
                return 'Medium MAE'
        
        country_mae_summary['mae_cluster_std'] = country_mae_summary['mae_ensemble'].apply(classify_by_std)
        
        print(f"      Low MAE threshold: < {mae_mean - 0.5 * mae_std:.2f}pp")
        print(f"      High MAE threshold: > {mae_mean + 0.5 * mae_std:.2f}pp")
        
        cluster_sizes = country_mae_summary['mae_cluster_std'].value_counts()
        for cluster, size in cluster_sizes.items():
            cluster_data = country_mae_summary[country_mae_summary['mae_cluster_std'] == cluster]
            print(f"      {cluster}: {size} countries, mean MAE = {cluster_data['mae_ensemble'].mean():.2f}pp")
        
        # Comparison of Methods
        print("\n   Method Comparison (Agreement Analysis):")
        
        # Create a comparison matrix
        methods = ['mae_cluster_kmeans', 'mae_cluster_hierarchical', 'mae_cluster_percentile', 'mae_cluster_std']
        if has_jenks:
            methods.insert(0, 'mae_cluster_jenks')
        
        from sklearn.metrics import adjusted_rand_score
        
        print("\n      Adjusted Rand Index (1.0 = perfect agreement):")
        for i, method1 in enumerate(methods):
            for method2 in methods[i+1:]:
                ari = adjusted_rand_score(
                    country_mae_summary[method1],
                    country_mae_summary[method2]
                )
                print(f"         {method1.replace('mae_cluster_', '')} vs {method2.replace('mae_cluster_', '')}: {ari:.3f}")
        
        # Consensus clustering (majority vote)
        print("\n   Consensus Clustering (Majority Vote):")
        
        # For each country, see what most methods classify it as
        cluster_cols = [col for col in country_mae_summary.columns if col.startswith('mae_cluster_')]
        
        def get_consensus(row):
            votes = row[cluster_cols].value_counts()
            return votes.index[0]  # Return most common classification
        
        country_mae_summary['mae_cluster_consensus'] = country_mae_summary.apply(get_consensus, axis=1)
        
        consensus_sizes = country_mae_summary['mae_cluster_consensus'].value_counts()
        print(f"\n      Consensus cluster sizes:")
        for cluster, size in consensus_sizes.items():
            cluster_data = country_mae_summary[country_mae_summary['mae_cluster_consensus'] == cluster]
            print(f"         {cluster}: {size} countries, mean MAE = {cluster_data['mae_ensemble'].mean():.2f}pp")
        
        # Show example countries from each consensus cluster
        print("\n   Example Countries by Consensus Cluster:")
        for cluster_name in ['Low MAE', 'Medium MAE', 'High MAE']:
            cluster_countries = country_mae_summary[
                country_mae_summary['mae_cluster_consensus'] == cluster_name
            ].sort_values('mae_ensemble')
            
            print(f"\n      {cluster_name} ({len(cluster_countries)} countries):")
            print(f"         Best 5: {', '.join(cluster_countries.head(5)['countrynew'].tolist())}")
            print(f"         Worst 5: {', '.join(cluster_countries.tail(5)['countrynew'].tolist())}")
            print(f"         MAE range: {cluster_countries['mae_ensemble'].min():.2f} - {cluster_countries['mae_ensemble'].max():.2f}pp")
        
        # Continent distribution by cluster
        print("\n   Continent Distribution by Consensus Cluster:")
        continent_cluster = pd.crosstab(
            country_mae_summary['mae_cluster_consensus'],
            country_mae_summary['continent'],
            normalize='index'
        ) * 100
        print(continent_cluster.round(1))
        
        # Save clustering results
        country_mae_summary.to_csv('mae_clustering_results.csv', index=False)
        print("\n   ✓ Saved mae_clustering_results.csv")
        
        # Create visualization of clusters
        print("\n   Creating cluster visualization...")
        
        fig, axes = plt.subplots(2, 2)
        fig.set_size_inches(16, 12)
        fig.set_dpi(100)
        
        # Plot 1: Histogram with cluster boundaries (Jenks or K-means)
        ax1 = axes[0, 0]
        cluster_col = 'mae_cluster_jenks' if has_jenks else 'mae_cluster_kmeans'
        method_name = 'Jenks Natural Breaks' if has_jenks else 'K-Means'
        
        for cluster in ['Low MAE', 'Medium MAE', 'High MAE']:
            data = country_mae_summary[country_mae_summary[cluster_col] == cluster]['mae_ensemble']
            ax1.hist(data, bins=15, alpha=0.6, label=cluster)
        
        ax1.set_xlabel('Mean Absolute Error (pp)', fontweight='bold')
        ax1.set_ylabel('Number of Countries', fontweight='bold')
        ax1.set_title(f'MAE Distribution by Cluster\n({method_name})', fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Consensus cluster visualization
        ax2 = axes[0, 1]
        cluster_order = ['Low MAE', 'Medium MAE', 'High MAE']
        cluster_means = [country_mae_summary[country_mae_summary['mae_cluster_consensus'] == c]['mae_ensemble'].mean() 
                        for c in cluster_order]
        cluster_counts = [len(country_mae_summary[country_mae_summary['mae_cluster_consensus'] == c]) 
                         for c in cluster_order]
        
        colors = ['#2ecc71', '#f39c12', '#e74c3c']  # Green, Orange, Red
        bars = ax2.bar(range(3), cluster_means, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
        
        # Add count labels on bars
        for i, (bar, count, mean) in enumerate(zip(bars, cluster_counts, cluster_means)):
            ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                    f'{mean:.1f}pp\n(n={count})', ha='center', va='bottom', fontweight='bold')
        
        ax2.set_xticks(range(3))
        ax2.set_xticklabels(cluster_order)
        ax2.set_ylabel('Mean Absolute Error (pp)', fontweight='bold')
        ax2.set_title('Consensus Cluster Means', fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='y')
        
        # Plot 3: Dendrogram (Hierarchical Clustering)
        ax3 = axes[1, 0]
        from scipy.cluster.hierarchy import dendrogram
        
        # Plot dendrogram
        dend = dendrogram(Z, ax=ax3, no_labels=True, color_threshold=0)
        ax3.set_xlabel('Countries', fontweight='bold')
        ax3.set_ylabel('Distance', fontweight='bold')
        ax3.set_title('Hierarchical Clustering Dendrogram', fontweight='bold')
        ax3.axhline(y=Z[-2, 2], color='r', linestyle='--', label='3-cluster cutoff')
        ax3.legend()
        
        # Plot 4: Continent composition of clusters
        ax4 = axes[1, 1]
        
        continent_cluster_counts = pd.crosstab(
            country_mae_summary['mae_cluster_consensus'],
            country_mae_summary['continent']
        )
        
        continent_cluster_counts.plot(kind='bar', stacked=True, ax=ax4, 
                                      colormap='tab10', edgecolor='black', linewidth=0.5)
        ax4.set_xlabel('Cluster', fontweight='bold')
        ax4.set_ylabel('Number of Countries', fontweight='bold')
        ax4.set_title('Continent Composition by Cluster', fontweight='bold')
        ax4.legend(title='Continent', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax4.set_xticklabels(ax4.get_xticklabels(), rotation=0)
        
        plt.tight_layout()
        plt.savefig('mae_clustering_visualization.png', dpi=300, bbox_inches='tight')
        plt.savefig('mae_clustering_visualization.pdf', dpi=300, bbox_inches='tight')
        print("   ✓ Saved mae_clustering_visualization.png")
        print("   ✓ Saved mae_clustering_visualization.pdf")
        plt.close()
    
else:
    print("   ⚠️  Continent data not available")


# ================================================================
# 8. Stage 5 Special Analysis (Willingness Data Issue)
# ================================================================

print("\n" + "="*80)
print("8. STAGE 5 ANALYSIS (OWN WILLINGNESS DATA)")
print("="*80)

if 'mean_own_willingness' in df.columns:
    stage5_df = df[df['stage'] == 5].copy()
    stage5_df['own_willingness_pct'] = stage5_df['mean_own_willingness'] * 100
    
    print("\n8.1 Are models echoing the own_willingness data?")
    for col in model_cols + ['pred_ensemble']:
        model_name = col.replace('pred_', '')
        
        # Correlation between prediction and own_willingness
        valid = stage5_df[[col, 'own_willingness_pct']].dropna()
        if len(valid) > 0:
            r, p = pearsonr(valid[col], valid['own_willingness_pct'])
            
            # Mean difference
            mean_diff = (valid[col] - valid['own_willingness_pct']).mean()
            
            print(f"\n   {model_name}:")
            print(f"      Correlation with own_willingness: r = {r:.3f} (p = {p:.4f})")
            print(f"      Mean prediction - own_willingness: {mean_diff:.2f}pp")
            
            if abs(r) > 0.9:
                print(f"      ⚠️  HIGH CORRELATION - may be echoing input!")
            elif mean_diff < -10:
                print(f"      ✓ Predicting lower (capturing PI effect)")
    
    print("\n8.2 Stage 5 vs Other Stages (for same countries):")
    stage1_pred = df[df['stage'] == 1].set_index('countrynew')['pred_ensemble']
    stage5_pred = df[df['stage'] == 5].set_index('countrynew')['pred_ensemble']
    common = stage1_pred.index.intersection(stage5_pred.index)
    
    diff_1_to_5 = (stage5_pred.loc[common] - stage1_pred.loc[common]).mean()
    print(f"   Mean change from Stage 1 to Stage 5: {diff_1_to_5:.2f}pp")

# ================================================================
# 9. Export Summary Statistics
# ================================================================

print("\n" + "="*80)
print("9. EXPORTING RESULTS")
print("="*80)

# Create summary stats file
summary_stats = {
    'Total Observations': len(df),
    'Countries': df['countrynew'].nunique(),
    'Stages': df['stage'].nunique(),
    'Models': len(model_cols)
}

# Save to file
with open('analysis_summary.txt', 'w') as f:
    f.write("LLM PLURALISTIC IGNORANCE PREDICTION ANALYSIS SUMMARY\n")
    f.write("="*80 + "\n\n")
    
    f.write("Dataset Overview:\n")
    for key, val in summary_stats.items():
        f.write(f"  {key}: {val}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("Mean Predictions by Stage:\n")
    f.write(stage_changes.to_string())
    
    if has_ground_truth:
        f.write("\n\n" + "="*80 + "\n")
        f.write("Model Accuracy (MAE):\n")
        f.write(mae_stats.to_string())
        
        f.write("\n\n" + "="*80 + "\n")
        f.write("Correlations with Ground Truth:\n")
        f.write(corr_df.to_string())
        
        if 'continent' in df.columns and df['continent'].notna().any():
            f.write("\n\n" + "="*80 + "\n")
            f.write("MAE by Continent:\n")
            f.write(continent_mae.to_string())
            
            f.write("\n\n" + "="*80 + "\n")
            f.write("Predictions by Continent:\n")
            f.write(continent_pred.to_string())

print("   ✓ Saved analysis_summary.txt")

# Save processed data with all computed metrics
df.to_csv('predictions_with_analysis.csv', index=False)
print("   ✓ Saved predictions_with_analysis.csv")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print("\nKey Files Created:")
print("  - analysis_summary.txt (summary statistics)")
print("  - predictions_with_analysis.csv (full data with computed metrics)")
if has_ground_truth and 'continent' in df.columns and df['continent'].notna().any():
    print("  - mae_by_continent_with_ci.png (visualization)")
    print("  - mae_by_continent_with_ci.pdf (visualization)")
    print("  - mae_by_continent_with_ci_stats.csv (detailed statistics)")
    print("  - mae_clustering_results.csv (country-level clustering by MAE)")
    print("  - mae_clustering_visualization.png (cluster analysis plots)")
    print("  - mae_clustering_visualization.pdf (cluster analysis plots)")
print("\nNext Steps:")
print("  1. Review the statistics above")
print("  2. Create visualizations (run 4_VISUALIZATIONS.py)")
print("  3. Statistical tests for stage effects")
print("  4. Write up results for paper")

LLM PLURALISTIC IGNORANCE PREDICTION ANALYSIS

1. Loading data...
   ✓ Loaded 1000 rows
   ✓ Countries: 125
   ✓ Stages: [1, 2, 3, 4, 5, 6, 7, 8]
   ✓ Models: ['gpt', 'claude', 'gemini', 'llama']
   ✓ Created ensemble prediction (mean of 4 models)
   ✓ All countries mapped to continents
   ✓ Continents: ['Africa', 'Asia', 'Europe', 'North America', 'Oceania', 'South America']
   ✓ Countries by continent:
      Africa: 29 countries
      Asia: 35 countries
      Europe: 38 countries
      North America: 11 countries
      Oceania: 2 countries
      South America: 10 countries

2. DESCRIPTIVE STATISTICS

2.1 Predictions by Stage:
      pred_gpt                    pred_claude                   pred_gemini  \
          mean    std   min   max        mean   std   min   max        mean   
stage                                                                         
1        39.76   7.24  30.0  50.0       35.52  3.47  22.5  45.2       31.55   
2        32.68   7.58  15.0  45.0       33.35  5

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>